In [ ]:
import pandas as pd

app = pd.read_parquet("../data/processed/appearances.parquet")
app.shape

In [ ]:
app = app.sort_values(["pitcher", "season", "game_date"])
g = app.groupby(["pitcher", "season"])

app["next_date"] = g["game_date"].shift(-1)
app["gap_after"] = (app["next_date"] - app["game_date"]).dt.days
app["days_rest"] = g["game_date"].diff().dt.days

In [ ]:
app["gap_after"].describe()
app["gap_after"].quantile([.5, .75, .9, .95, .99])
app["gap_after"].clip(upper=40).hist(bins=40)

In [ ]:
# --- 1. Starter versus reliever -------------------------------------
# A starter's median outing is far longer than a reliever's.
med_pitches = g["n_pitches"].transform("median")
app["is_starter"] = (med_pitches >= 50).astype(int)

# Sanity check: does this split look like two populations?
from IPython.display import display

display(app.groupby("is_starter")["n_pitches"].describe())
display(app.groupby("is_starter")["days_rest"].median())

In [ ]:
# --- 2. Closer-usage components -------------------------------------
# Closers are characterized by three things, none of which is labeled
# in the data: they enter late, they throw short outings, and they are
# used on back-to-back days.

app["entered_ninth"] = (app["first_inning"] >= 9).astype(int)
app["short_outing"]  = (app["n_pitches"] <= 20).astype(int)
app["back_to_back"]  = (app["days_rest"] == 1).astype(int)

In [ ]:
# Season-level usage profile, one row per pitcher-season
usage = (app.groupby(["pitcher", "season"], as_index=False)
            .agg(ninth_rate = ("entered_ninth", "mean"),
                 short_rate = ("short_outing", "mean"),
                 b2b_rate   = ("back_to_back", "mean"),
                 n_apps     = ("n_pitches", "size")))

# only meaningful for relievers with enough appearances
rel = usage[(usage["n_apps"] >= 15)].copy()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

cols = ["ninth_rate", "short_rate", "b2b_rate"]

pca_pipe = Pipeline([("scale", StandardScaler()),
                     ("pca",   PCA(n_components=1))])

rel["closer_index"] = pca_pipe.fit_transform(rel[cols]).ravel()

# check the loadings, exactly as you would read an ICA component
loadings = pd.Series(pca_pipe.named_steps["pca"].components_[0], index=cols)
print(loadings)
print("variance explained:", pca_pipe.named_steps["pca"].explained_variance_ratio_)

In [ ]:
top20 = rel.sort_values("closer_index", ascending=False).head(20)
top20 = top20.merge(app[["pitcher","player_name"]].drop_duplicates(), on="pitcher")
top20[["player_name","season","closer_index","ninth_rate","short_rate","b2b_rate"]]

In [ ]:
GAP_THRESHOLD = 15     # justify from your histogram

# The final appearance of a pitcher-season has no observable gap
app = app[app["next_date"].notna()].copy()

app["absence"] = (app["gap_after"] >= GAP_THRESHOLD).astype(int)

print("overall:", app["absence"].mean())
print(app.groupby("is_starter")["absence"].mean())

In [ ]:
display(app[app["is_starter"] == 0]["gap_after"].quantile([.5, .75, .9, .95, .99]))
display(app[app["is_starter"] == 1]["gap_after"].quantile([.5, .75, .9, .95, .99]))

In [ ]:
import numpy as np

STARTER_THRESHOLD  = 9
RELIEVER_THRESHOLD = 11

app["absence"] = np.where(
    app["is_starter"] == 1,
    (app["gap_after"] >= STARTER_THRESHOLD).astype(int),
    (app["gap_after"] >= RELIEVER_THRESHOLD).astype(int)
)

print("overall:", app["absence"].mean())
print(app.groupby("is_starter")["absence"].mean())

In [ ]:
app.to_parquet("../data/processed/appearances_with_outcome.parquet")